In [ ]:
# plot_her_mpt.py
# Requirements: pandas, matplotlib
# Usage: edit `filepath` to point at your .mpt (or run in same directory)

import os
import re
import pandas as pd
import matplotlib.pyplot as plt

def read_ec_lab_mpt(filepath):
    """
    Robust reader for EC-Lab ASCII (.mpt) text files.
    Detects the header line containing 'Ewe/V' and '<I>/mA',
    then parses the following lines as tab-separated data and
    returns a DataFrame with numeric columns.
    """
    with open(filepath, 'r', encoding='latin1', errors='ignore') as fh:
        lines = fh.readlines()

    # Find header line index (contains both Ewe/V and <I>/mA)
    header_idx = None
    for i, line in enumerate(lines[:400]):   # header usually near top
        if 'Ewe/V' in line and '<I>/mA' in line:
            header_idx = i
            break
    if header_idx is None:
        raise ValueError("Header with 'Ewe/V' and '<I>/mA' not found in file: " + filepath)

    # Build header names and read data after header row
    header_names = [h.strip() for h in lines[header_idx].strip().split('\t')]
    # Some files may have extra empty column names - we keep them as-is
    df = pd.read_csv(filepath,
                     sep='\t',
                     names=header_names,
                     skiprows=header_idx+1,
                     engine='python',
                     encoding='latin1',
                     comment='#')

    # Extract required columns (case-insensitive match)
    col_map = {col: re.sub(r'[^0-9a-zA-Z]', '', col).lower() for col in df.columns}
    pot_col = None
    cur_col = None
    for col, norm in col_map.items():
        if 'ewe' in norm or 'potential' in norm or 'ewev' in norm:
            pot_col = col
        if 'current' in norm or 'i' == norm or 'ima' in norm or 'ma' in norm or 'i' in norm:
            # choose a candidate for current; refine below
            if '<i>' in col.lower() or '<i>/ma' in col.lower() or 'ma' in norm:
                cur_col = col
            elif cur_col is None:
                cur_col = col

    # Final fallback: use exact column names if present
    if pot_col is None and 'Ewe/V' in df.columns:
        pot_col = 'Ewe/V'
    if cur_col is None and '<I>/mA' in df.columns:
        cur_col = '<I>/mA'

    if pot_col is None or cur_col is None:
        # As last resort, try to detect numeric ranges: potentials ~ -5..+5, currents often small
        for col in df.columns:
            ser = pd.to_numeric(df[col], errors='coerce')
            if ser.notna().sum() < 5:
                continue
            mn, mx = ser.min(), ser.max()
            # potential heuristic
            if pot_col is None and -5 <= mn <= 5 and -5 <= mx <= 5:
                pot_col = col
            elif cur_col is None:
                cur_col = col

    if pot_col is None or cur_col is None:
        raise ValueError("Could not identify potential/current columns automatically.")

    # Clean numeric columns
    pot = pd.to_numeric(df[pot_col], errors='coerce')
    cur = pd.to_numeric(df[cur_col], errors='coerce')
    out = pd.DataFrame({'Potential_V': pot, 'Current_mA': cur}).dropna().reset_index(drop=True)
    return out

def plot_current_vs_potential(df, title=None, savepath=None):
    """
    Plots Current (mA) vs Potential (V) with professional formatting.
    Does not set explicit colors so default matplotlib colors are used.
    """
    plt.rcParams.update({'font.size': 12})
    fig, ax = plt.subplots(figsize=(8,6))
    ax.plot(df['Potential_V'], df['Current_mA'], marker='o', linestyle='-', markersize=3, linewidth=0.9)
    ax.set_xlabel('Potential (E$_{we}$) / V', fontsize=13)
    ax.set_ylabel('Current (<I>) / mA', fontsize=13)
    if title is None:
        title = 'HER — Current vs Potential'
    ax.set_title(title, fontsize=14, weight='bold')
    ax.grid(True, which='both', linestyle='--', linewidth=0.5)
    ax.tick_params(axis='both', which='major', labelsize=11)
    plt.tight_layout()
    if savepath:
        fig.savefig(savepath, dpi=300)
    return fig, ax

if __name__ == "__main__":
    # === USER CONFIGURE ===
    # Path to your .mpt file (change this to the file you want to plot)
    filepath = "/path/to/your/file.mpt"   # <<-- edit this
    save_plot_to = "HER_Current_vs_Potential.png"  # edit or None
    # ======================

    # Read file
    df = read_ec_lab_mpt(filepath)
    # Show preview (first rows)
    print("Data preview (first 8 rows):")
    print(df.head(8).to_string(index=False))

    # Plot and save
    fig, ax = plot_current_vs_potential(df, title=os.path.basename(filepath), savepath=save_plot_to)
    print(f"Plot saved to: {os.path.abspath(save_plot_to)}")
    plt.show()

: 